# Naive VascX baseline — 순수 VascX 만 돌리기

**목적**: `01-registration.ipynb` 의 정렬 파이프라인이 실제로 fluctuation 을 줄이는지 확인.

**Naive 정의** = **VascX 그대로**:
- Base preprocess (fundus square crop) — 원본 → 512 정사각. 이게 유일한 전처리
- VascX mask 추론 → av / vessels / discs
- Feature = mask 픽셀 수 / **fundus 유효 픽셀 수** (per-image, 이미지마다 다름)
- **Registration 없음, 고정 ROI 없음, common_valid 없음**

**공통**:
- 같은 환자 2 명, 같은 이미지, 같은 VascX 마스크 (`01` 캐시 재사용)
- 같은 laterality 판정 (disc mask centroid)
- 같은 feature 함수 (`vessel_features`)

**비교**: 두 파이프라인의 timeline / std / CV 를 나란히 표시.

## 0. Imports + 경로


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi

SQUARE_SIZE = 512

src_path      = Path("g:/RAG/20230118_HCS_Fundus_image")
csv_path      = Path("G:/RAG/longitudinal/csv/longitudinal_final_260820_reclassified.csv")
prep_root     = Path(f"g:/RAG/project_1/output/preprocessed_{SQUARE_SIZE}") / src_path.name
rgb_path      = prep_root / "rgb"
out_root      = Path(f"g:/RAG/project_1/output/vascx_output_{SQUARE_SIZE}") / src_path.name
av_path       = out_root / "av"
vessels_path  = out_root / "vessels"
discs_path    = out_root / "discs"
aligned_root  = Path(f"g:/RAG/project_1/output/aligned_{SQUARE_SIZE}") / src_path.name

# naive 산출물 저장 위치 (aligned 랑 구분)
naive_root = Path(f"g:/RAG/project_1/output/naive_{SQUARE_SIZE}") / src_path.name
naive_root.mkdir(parents=True, exist_ok=True)
print("naive out:", naive_root)


## 1. 같은 환자 2 명 로드 (01 과 동일 방식)


In [ ]:
data = pd.read_csv(csv_path, low_memory=False)
counts = data.groupby("ID").size().sort_values(ascending=False)
TOP_N = 3
patient_ids = counts.head(TOP_N).index.tolist()

cohort = data[data["ID"].isin(patient_ids)].copy()
cohort["LAB_DTM"] = pd.to_datetime(cohort["LAB_DTM"])
cohort = cohort.sort_values(["ID", "LAB_DTM", "shot"]).reset_index(drop=True)
print(f"환자 {patient_ids}, 총 {len(cohort)} 장")


## 2. 인벤토리 — 좌우안 판정 (`01` 과 동일: disc mask centroid)

01 노트북에서 저장한 aligned mask 는 warp 됐으니, 여기서는 **base disc mask** 를 다시 읽어서 centroid 계산.


In [ ]:
MIN_SEP_FRAC = 0.02
MIN_DISC_PX  = 50

def disc_geometry(disc_mask, min_area_px=MIN_DISC_PX):
    m = disc_mask.astype(bool)
    if m.sum() < min_area_px:
        return dict(cx=np.nan, cy=np.nan, diameter=np.nan, area_px=float(m.sum()), found=False)
    lab, n = ndi.label(m)
    if n > 1:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = lab == (int(np.argmax(sizes)) + 1)
    area = float(m.sum())
    cy, cx = ndi.center_of_mass(m)
    return dict(cx=float(cx), cy=float(cy),
                diameter=2.0 * np.sqrt(area / np.pi),
                area_px=area, found=True)

# fovea 좌표는 01 에서 이미 저장 안 했으므로 새로 뽑아야 함. 하지만 laterality 만 필요하고
# 01 의 inv_df 를 그대로 재사용해도 됨 → 있는지 확인, 없으면 재추론
inv_df_path = aligned_root / "registration_log.csv"
if inv_df_path.exists():
    reg_log = pd.read_csv(inv_df_path)
    print(f"01 결과 로드: {len(reg_log)} rows")
else:
    raise RuntimeError("01-registration.ipynb 먼저 실행해서 registration_log.csv 를 만드세요.")

# reg_log 에 이미 patient/eye/date/id 들어있음 → 그대로 사용
reg_log["date"] = pd.to_datetime(reg_log["date"])
print(reg_log[["id", "patient", "eye", "date"]].head())


In [ ]:
# 인벤토리 dict 재구성
inventory = {}
for (pid, eye), grp in reg_log.groupby(["patient", "eye"], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.sort_values(["date", "shot"]).to_dict("records")

for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        print(f"{pid}/{eye}: {len(visits)} visits")


## 3. Naive ROI = 각 이미지의 fundus 전체 (per-image)

고정 원형 ROI **없음**. 각 이미지의 fundus 픽셀 (`rgb.sum(-1) > 10`) 이 그대로 denominator.  
촬영마다 fundus 크기가 조금씩 다르므로 denominator 도 이미지마다 다름 — 그게 naive 파이프라인의 특징.

In [ ]:
def fundus_valid(rgb):
    """검은 배경 (fundus 밖) 제외, per-image mask."""
    return rgb.sum(-1) > 10

# 확인: 몇 개 이미지 fundus area 편차 확인
sample_ids = list(inventory[patient_ids[0]].values())[0][:5]
areas = [fundus_valid(np.array(Image.open(rgb_path / f"{v['id']}.png"))).sum()
         for v in sample_ids]
print(f"fundus_valid area (px) 샘플 5개: {areas}")
print(f"편차: {max(areas)-min(areas)} px ({100*(max(areas)-min(areas))/np.mean(areas):.1f}%)")

## 4. Feature 계산 — base mask + per-image fundus_valid

In [ ]:
def vessel_features(vessel_mask, av_mask, disc_mask, roi):
    roi_n = max(int(roi.sum()), 1)
    v_in  = (vessel_mask > 0) & roi
    art   = np.isin(av_mask, (1, 3)) & roi
    vei   = np.isin(av_mask, (2, 3)) & roi
    disc_in = (disc_mask > 0) & roi
    return {
        "roi_area_px":    roi_n,
        "vessel_area_px": int(v_in.sum()),
        "vessel_density": v_in.sum() / roi_n,
        "artery_density": art.sum()  / roi_n,
        "vein_density":   vei.sum()  / roi_n,
        "AV_ratio_area":  art.sum() / max(int(vei.sum()), 1),
        "disc_area_px":   int(disc_in.sum()),
        "disc_frac":      disc_in.sum() / roi_n,
    }

feat_rows = []
for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye == "unknown": continue
        for v in visits:
            vid = v["id"]
            try:
                rgb  = np.array(Image.open(rgb_path      / f"{vid}.png"))
                vs_m = np.array(Image.open(vessels_path / f"{vid}.png"))
                av_m = np.array(Image.open(av_path      / f"{vid}.png"))
                dc_m = np.array(Image.open(discs_path   / f"{vid}.png"))
            except FileNotFoundError:
                continue
            roi = fundus_valid(rgb)              # ← per-image fundus
            feats = vessel_features(vs_m, av_m, dc_m, roi)
            feat_rows.append({
                "id": vid, "patient": pid, "eye": eye,
                "date": v["date"], "shot": v["shot"],
                **feats,
            })

features_naive = pd.DataFrame(feat_rows).sort_values(["patient", "eye", "date"]).reset_index(drop=True)
features_naive.to_csv(naive_root / "features_naive.csv", index=False)
print(f"naive features 저장: {len(features_naive)} rows")
features_naive.head()

### 4-viz-1. Naive RGB timeline — base 이미지 그대로

01 노트북 timeline 의 첫 번째 열 (original) 과 완전히 동일. 아무런 warp 도 ROI 도 적용 안 함.

In [ ]:
def plot_naive_rgb_timeline(patient, eye, visits):
    """base RGB 그대로 (warp X, ROI X, overlay X)."""
    n = len(visits)
    fig, axes = plt.subplots(1, n, figsize=(2.4*n, 2.6))
    if n == 1: axes = [axes]
    for i, v in enumerate(visits):
        rgb = np.array(Image.open(rgb_path / f"{v['id']}.png"))
        axes[i].imshow(rgb)
        axes[i].set_title(pd.Timestamp(v["date"]).strftime('%Y-%m-%d'), fontsize=8)
        axes[i].axis("off")
    fig.suptitle(f"Patient {patient} — {eye} eye — NAIVE base RGB (warp X, ROI X)",
                 fontsize=11)
    plt.tight_layout(); plt.show()

for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye == "unknown": continue
        plot_naive_rgb_timeline(pid, eye, visits)

### 4-viz-2. Naive vessel mask timeline — fundus 전체 위에 vessel 오버레이

원형 ROI 없음. Base RGB 위에 vessel mask 를 초록으로 오버레이. 제목에 `vessel_density` (median ±15% 이상은 빨간색 = spike).

In [ ]:
def plot_naive_vessel_timeline(patient, eye, visits):
    """base RGB + base vessel mask (warp X, ROI X)."""
    n = len(visits)
    sub = features_naive[(features_naive["patient"]==patient) & (features_naive["eye"]==eye)]
    dens_map = dict(zip(sub["id"], sub["vessel_density"]))
    d_median = sub["vessel_density"].median()

    fig, axes = plt.subplots(1, n, figsize=(2.4*n, 2.6))
    if n == 1: axes = [axes]

    for i, v in enumerate(visits):
        vid = v["id"]
        try:
            rgb  = np.array(Image.open(rgb_path      / f"{vid}.png"))
            mask = np.array(Image.open(vessels_path / f"{vid}.png"))
        except FileNotFoundError:
            axes[i].axis("off"); continue

        # base RGB 를 좀 어둡게 + vessel 은 초록으로 강조
        disp = (rgb.astype(np.float32) * 0.5).astype(np.uint8)
        disp[mask > 0] = [0, 255, 0]
        axes[i].imshow(disp)

        d = dens_map.get(vid, np.nan)
        rel = (d - d_median) / d_median if d_median else 0
        color = "red" if abs(rel) > 0.15 else ("orange" if abs(rel) > 0.08 else "black")
        axes[i].set_title(f"{pd.Timestamp(v['date']).strftime('%Y-%m-%d')}\nvd={d:.3f}",
                          fontsize=7, color=color)
        axes[i].axis("off")

    fig.suptitle(
        f"Patient {patient} — {eye} eye — NAIVE vessel mask (warp X, ROI X)  "
        f"median vd = {d_median:.3f}",
        fontsize=11,
    )
    plt.tight_layout(); plt.show()

for pid in patient_ids:
    for eye, visits in inventory.get(pid, {}).items():
        if eye == "unknown": continue
        plot_naive_vessel_timeline(pid, eye, visits)

## 5. 요약

- **`features_naive`** (`naive_root/features_naive.csv`) — 순수 VascX 로 뽑은 feature
  - Base preprocess → mask 추론 → per-image fundus 를 denominator 로 density 계산
  - Warp X, ROI 원 X, common_valid X
- **Naive vs Aligned 비교**는 `02-compare.ipynb` 에서.
